# 🌍 AirLyst: [01] Data Acquisition & Merging
**Mission: Fetch & Combine**

This notebook focuses strictly on fetching 1.5+ years of historical data from the Open-Meteo APIs (Air Quality and Weather) and merging them into a single raw dataset.


In [1]:
import sys
import os
from pathlib import Path
import pandas as pd
import requests

# Setup Path
ROOT_DIR = Path("C:/Users/Dell/Desktop/AirLyst")
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

from backend.config.settings import settings

print(f" System Ready: {settings.APP_NAME}")
print(f" Target: {settings.CITY} (Lat: {settings.LATITUDE}, Lon: {settings.LONGITUDE})")
print(f" Target Window: {settings.START_DATE} to {settings.END_DATE}")

 System Ready: AirLyst AQI Predictor
 Target: Islamabad (Lat: 33.72, Lon: 73.04)
 Target Window: 2024-11-13 to 2026-05-15


## 📡 1. Fetching Raw Data
We use the dynamic dates defined in our global `settings`.

In [2]:
def fetch_data(url, params, name):
    print(f" Fetching {name} data...")
    response = requests.get(url, params=params)
    if response.status_code == 200:
        return pd.DataFrame(response.json()["hourly"])
    else:
        print(f" ERROR: {name} API failed.")
        return None

aqi_params = {
    "latitude": settings.LATITUDE,
    "longitude": settings.LONGITUDE,
    "hourly": "pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,us_aqi",
    "start_date": settings.START_DATE,
    "end_date": settings.END_DATE,
    "timezone": "auto"
}

weather_params = {
    "latitude": settings.LATITUDE,
    "longitude": settings.LONGITUDE,
    "hourly": "temperature_2m,relative_humidity_2m,precipitation,surface_pressure,wind_speed_10m,wind_direction_10m",
    "start_date": settings.START_DATE,
    "end_date": settings.END_DATE,
    "timezone": "auto"
}

df_aqi = fetch_data(settings.AIR_URL, aqi_params, "Air Quality")
df_weather = fetch_data(settings.WEATHER_URL, weather_params, "Weather")

 Fetching Air Quality data...
 Fetching Weather data...


## 🔗 2. Merging & Saving Raw Data

In [3]:
if df_aqi is not None and df_weather is not None:
    df_aqi['time'] = pd.to_datetime(df_aqi['time'])
    df_weather['time'] = pd.to_datetime(df_weather['time'])
    
    df_raw = pd.merge(df_aqi, df_weather, on="time", how="inner")
    
    out_dir = Path(ROOT_DIR / "backend/data")
    out_dir.mkdir(parents=True, exist_ok=True)
    
    raw_path = out_dir / "raw_historical_data.csv"
    df_raw.to_csv(raw_path, index=False)
    
    print(f" SUCCESS: Raw data saved to {raw_path}")
    print(f" Total Rows: {len(df_raw)}")
    display(df_raw.head())
else:
    print(" FAIL: Data acquisition failed.")

 SUCCESS: Raw data saved to C:\Users\Dell\Desktop\AirLyst\backend\data\raw_historical_data.csv
 Total Rows: 13176


,time,pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,us_aqi,temperature_2m,relative_humidity_2m,precipitation,surface_pressure,wind_speed_10m,wind_direction_10m
0,2024-11-13 00:00:00,108.7,104.1,965.0,79.9,11.2,3.0,151,16.5,63,0.0,947.6,6.0,333
1,2024-11-13 01:00:00,96.6,92.9,674.0,70.1,8.2,3.0,152,15.9,66,0.0,947.0,5.5,337
2,2024-11-13 02:00:00,84.5,81.3,477.0,61.4,5.9,4.0,153,16.0,64,0.0,946.5,5.6,335
3,2024-11-13 03:00:00,73.3,70.5,378.0,53.5,4.5,4.0,154,16.4,61,0.0,946.2,4.8,335
4,2024-11-13 04:00:00,63.1,60.8,372.0,46.8,3.9,5.0,155,17.3,53,0.0,946.1,2.7,337
